In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import OllamaLLM
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [ ]:
loader = PyPDFLoader('data\cv.pdf')
docs = loader.load()
docs

In [ ]:
#split document
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100, separators=['\n'])
docs = splitter.split_documents(docs)
docs

## Ollama Models

In [ ]:
#embedding 
embedding_model = OllamaEmbeddings(model='llama3.2')
#vectorstoredb 
faiss_db = FAISS.from_documents(docs, embedding_model)
faiss_db

In [ ]:
result = faiss_db.similarity_search_with_score('the job needs Python, Sql, Machine Learning and work experience')
result[-1]

### Retriever

In [ ]:
#create prompt template
prompt = ChatPromptTemplate([
    ("system", "You are a professional HR assistant helping evaluate how well a CV fits a job description."),
    ("human", 
     "Job Requirement:\n{input}\n\n"
     "Relevant Content from the CV:\n{context}\n\n"
     "Analyze the match and list strengths, gaps, and a recommendation.")
])

In [ ]:
llm = OllamaLLM(model='llama3.2', temperature=0)

In [ ]:
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

In [ ]:
retriever = faiss_db.as_retriever()
retriever_chain = create_retrieval_chain(retriever, document_chain)
retriever_chain

In [ ]:
result = retriever_chain.invoke({'input' : """

Bachelor?s/Master?s/Ph.D. in a quantitative field (Computer Science, Data Science, Statistics, Economics, Finance, or related, 3+ years of experience in data science, preferably with a focus on credit risk modeling, loan portfolio analytics, or financial risk assessment,
Strong knowledge of credit risk/anti-fraud in the consumer lending domain,
Expertise in Python and machine learning/statistics libraries (scikit-learn, LGBM, XGBoost, etc),
Proficiency in SQL and data processing libraries for analyzing large-scale credit and transaction datasets,
Experience with cloud-based technologies (GCP preferred) and Docker for scalable model deployment,
Ability to translate business objectives into data-driven risk strategies and communicate findings effectively,

"""
})
result['answer']


## HuggingFace Models

In [ ]:
import os 

In [ ]:
load_dotenv()
os.environ['huggingfacehub_api_token'] = os.getenv('huggingfacehub_api_token')

In [ ]:
huggingfacehub_api_token = os.getenv('huggingfacehub_api_token')

In [ ]:
#load embedding hf model 
hf_embedding_model = HuggingFaceEmbeddings(model='all-MiniLM-L6-v2')

In [ ]:
hf_faiss_db = FAISS.from_documents(docs, hf_embedding_model)

In [ ]:
result = hf_faiss_db.similarity_search('the job needs Python, Sql, Machine Learning and work experience')
result

### Retriever

In [ ]:
from huggingface_hub import InferenceClient
from langchain_community.llms import HuggingFaceHub
from transformers import pipeline 

In [ ]:
from huggingface_hub import InferenceClient

# Ganti 'your_api_token' dengan token API Anda
client = InferenceClient(token=huggingfacehub_api_token)

# Jalankan inferensi
response = client.text_generation(
    model="microsoft/Phi-4-reasoning-plus",
    inputs="Who are you?",
    parameters={"max_new_tokens": 50}
)

print(response)

In [ ]:
input = 2

def inputData(input:int) : 
    #if not isinstance(input, str) :
     #   raise ValueError('data must be string') 
    print(input)

type(inputData(input))   